# groupby, pivot_table and value_counts

**Video 6.5** | ShopKart project

**Goal:** Summarise 5,000 cleaned orders by city, payment mode and month - the pandas version of SUMIFS, pivot tables and SQL GROUP BY.

### How to open in Colab
1. Go to colab.research.google.com → **File → Upload notebook** → choose this `.ipynb` file.
2. Click the **folder icon** on the left sidebar (Files panel) and drag in the data files: `orders_clean.csv`.
3. Run cells one by one with **Shift + Enter**.

> Files you upload to the Files panel disappear when the runtime resets. If you see `FileNotFoundError`, upload them again.

In [1]:
import pandas as pd
df = pd.read_csv("orders_clean.csv", parse_dates=["order_date"])
df["revenue"] = (df["quantity"] * df["unit_price"] * (1 - df["discount"])).round(2)
df["order_month"] = df["order_date"].dt.month
df["is_returned"] = (df["returned"] == "Y").astype(int)
df.shape

(5000, 15)

## 1. value_counts: how many of each
| Excel | SQL | pandas |
|---|---|---|
| COUNTIF | `SELECT city, COUNT(*) ... GROUP BY city` | `df["city"].value_counts()` |

In [2]:
df["city"].value_counts()

city
Delhi        804
Mumbai       759
Bengaluru    736
Hyderabad    495
Pune         451
Chennai      442
Kolkata      414
Lucknow      326
Ahmedabad    303
Jaipur       270
Name: count, dtype: int64

In [3]:
df["payment_mode"].value_counts(normalize=True).round(3) * 100   # as % of orders

payment_mode
UPI            41.1
COD            19.7
Credit Card    17.8
Debit Card     13.7
Net Banking     5.9
Unknown         1.8
Name: proportion, dtype: float64

## 2. groupby: split → apply → combine
Split rows into groups (one per city), apply a calculation (sum of revenue), combine into one result.

In [4]:
city_revenue = df.groupby("city")["revenue"].sum().sort_values(ascending=False).round(0)
city_revenue

city
Delhi        2323405.0
Bengaluru    2256613.0
Mumbai       2090206.0
Hyderabad    1494617.0
Pune         1387698.0
Chennai      1333559.0
Kolkata      1237204.0
Ahmedabad    1065043.0
Lucknow       791224.0
Jaipur        725080.0
Name: revenue, dtype: float64

## 3. Several numbers at once with agg

In [5]:
city_summary = df.groupby("city").agg(
    orders=("order_id", "count"),
    revenue=("revenue", "sum"),
    avg_order=("revenue", "mean"),
    avg_rating=("rating", "mean"),
    return_rate=("is_returned", "mean"),
).round(2).sort_values("revenue", ascending=False)
city_summary

,orders,revenue,avg_order,avg_rating,return_rate
city,,,,,
Delhi,804,2323405.40,2889.81,4.04,0.10
Bengaluru,736,2256613.25,3066.05,4.07,0.12
Mumbai,759,2090206.00,2753.89,4.04,0.09
Hyderabad,495,1494616.75,3019.43,4.05,0.13
Pune,451,1387698.30,3076.94,3.47,0.32
Chennai,442,1333558.90,3017.10,4.06,0.13
Kolkata,414,1237204.20,2988.42,4.05,0.14
Ahmedabad,303,1065043.05,3514.99,4.00,0.14
Lucknow,326,791223.65,2427.07,3.74,0.13


Look at **Pune's** return rate compared with every other city. We will dig into this in 6.7 and Module 9.

In [6]:
print("Overall return rate:", round(df["is_returned"].mean() * 100, 1), "%")
print("Pune return rate:", round(df.loc[df["city"] == "Pune", "is_returned"].mean() * 100, 1), "%")

Overall return rate: 13.7 %
Pune return rate: 32.4 %


## 4. Group by two columns

In [7]:
df.groupby(["city", "payment_mode"])["is_returned"].mean().round(3).head(12)

city       payment_mode
Ahmedabad  COD             0.182
           Credit Card     0.174
           Debit Card      0.143
           Net Banking     0.077
           UPI             0.129
           Unknown         0.000
Bengaluru  COD             0.200
           Credit Card     0.100
           Debit Card      0.067
           Net Banking     0.113
           UPI             0.103
           Unknown         0.429
Name: is_returned, dtype: float64

## 5. pivot_table: the Excel pivot table in one line

In [8]:
pivot = pd.pivot_table(df, index="city", columns="payment_mode", values="revenue",
                       aggfunc="sum", margins=True, margins_name="Total").round(0)
pivot

payment_mode,COD,Credit Card,Debit Card,Net Banking,UPI,Unknown,Total
city,,,,,,,
Ahmedabad,236413.0,92428.0,128615.0,94966.0,484479.0,28144.0,1065043.0
Bengaluru,419469.0,375289.0,387248.0,109775.0,952811.0,12020.0,2256613.0
Chennai,324441.0,266825.0,168059.0,47848.0,518751.0,7635.0,1333559.0
Delhi,466115.0,460627.0,351478.0,66148.0,931592.0,47445.0,2323405.0
Hyderabad,378534.0,191763.0,170256.0,120813.0,609635.0,23616.0,1494617.0
Jaipur,137558.0,150247.0,127409.0,37899.0,256146.0,15820.0,725080.0
Kolkata,207092.0,212154.0,152745.0,119959.0,529742.0,15512.0,1237204.0
Lucknow,138359.0,143614.0,108941.0,68254.0,308907.0,23148.0,791224.0
Mumbai,410013.0,326890.0,463108.0,95151.0,715438.0,79605.0,2090206.0


In [9]:
# Monthly revenue - a trend we will chart in Module 8
monthly = df.groupby("order_month")["revenue"].sum().round(0)
monthly

order_month
1     1073267.0
2      863480.0
3     1080122.0
4      983481.0
5      786912.0
6      793841.0
7      873632.0
8     1203393.0
9     1543639.0
10    1978317.0
11    1965664.0
12    1558902.0
Name: revenue, dtype: float64

In [10]:
# Return rate by payment mode, as a pivot with counts too
pd.pivot_table(df, index="payment_mode", values="is_returned", aggfunc=["mean", "count"]).round(3)

,mean,count
,is_returned,is_returned
payment_mode,,
COD,0.191,983
Credit Card,0.126,888
Debit Card,0.123,685
Net Banking,0.131,297
UPI,0.121,2057
Unknown,0.167,90


## Practice
1. Which payment mode has the highest average delivery_days?
2. Make a pivot of average rating with rows = city and columns = returned (Y/N).

In [11]:
print(df.groupby("payment_mode")["delivery_days"].mean().round(2).sort_values(ascending=False))
pd.pivot_table(df, index="city", columns="returned", values="rating", aggfunc="mean").round(2)

payment_mode
UPI            4.18
Unknown        4.17
Net Banking    4.15
Credit Card    4.14
COD            4.13
Debit Card     4.12
Name: delivery_days, dtype: float64


returned,N,Y
city,,
Ahmedabad,4.19,2.83
Bengaluru,4.22,2.94
Chennai,4.24,2.89
Delhi,4.19,2.73
Hyderabad,4.23,2.81
Jaipur,3.96,2.62
Kolkata,4.23,3.00
Lucknow,3.88,2.74
Mumbai,4.16,2.83
